# 03 · Error analysis, thresholds and unintended bias

A leaderboard says which model wins. This notebook says **what the winner still
gets wrong**, and whether it is wrong in a way that harms a particular group.
Run `02` first — this one reuses its out-of-fold predictions.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src.config import Config

# `memory` keeps this session's intermediate frames out of the file system;
# switch to "file" to persist them under data/processed/.
CONFIG = Config(persistence_backend="memory")
CONFIG.ensure_directories()
CONFIG.label_columns

In [ ]:
from src.dataio import load_training_data, raw_files_available

# allow_synthetic keeps the notebook runnable before the Kaggle download;
# every number below is only meaningful once raw_files_available() is True.
train = load_training_data(config=CONFIG, allow_synthetic=True)
print("real dataset:", raw_files_available(), "| rows:", len(train))
train.head(3)

In [ ]:
from src.dataio import label_matrix, stratified_subsample, texts
from src.evaluation import cross_validate_model, make_multilabel_folds
from src.models import build_model

SAMPLE_SIZE = 20_000
frame = stratified_subsample(train, SAMPLE_SIZE, config=CONFIG) if len(train) > SAMPLE_SIZE else train
comment_text = texts(frame, CONFIG).tolist()
y = label_matrix(frame, CONFIG)
label_names = list(CONFIG.label_columns)

folds, strategy = make_multilabel_folds(y, n_splits=CONFIG.n_splits,
                                        random_state=CONFIG.random_state)
result = cross_validate_model(
    model_factory=lambda: build_model("nbsvm", label_names=label_names),
    texts=comment_text, y=y, label_names=label_names, folds=folds, strategy=strategy,
)
oof = result.oof_scores          # every row scored by the fold that did not train on it
result.summary.round(4)

## Why 0.5 is the wrong threshold

The scores of a rare label live far below 0.5, so the default cut-off predicts
almost nothing positive. Sweeping the threshold shows how much F1 is left on
the table.

In [ ]:
from sklearn.metrics import f1_score

grid = np.linspace(0.01, 0.99, 99)
fig, ax = plt.subplots(figsize=(7.5, 4))
for index, label in enumerate(label_names):
    curve = [f1_score(y[:, index], (oof[:, index] >= cut).astype(int), zero_division=0)
             for cut in grid]
    ax.plot(grid, curve, label=label)
    ax.scatter([grid[int(np.argmax(curve))]], [max(curve)], s=25, zorder=3)
ax.axvline(0.5, color="grey", ls="--", lw=1, label="default 0.5")
ax.set(xlabel="threshold", ylabel="F1", title="F1 against decision threshold (out-of-fold)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "threshold_sweep.png", dpi=200)

In [ ]:
from src.evaluation import describe_thresholds, optimise_thresholds, per_label_metrics

tuned = optimise_thresholds(y, oof)
print("tuned thresholds:", describe_thresholds(tuned, label_names))

default_metrics = per_label_metrics(y, oof, label_names, thresholds=0.5)["f1"]
tuned_metrics = per_label_metrics(y, oof, label_names, thresholds=tuned)["f1"]
pd.DataFrame({"F1 @ 0.5": default_metrics, "F1 @ tuned": tuned_metrics,
              "gain": tuned_metrics - default_metrics}).round(4)

## Confusion per label

Where the errors sit: false positives annoy honest editors, false negatives let
abuse through. The two are not equally costly, and the threshold is the dial
that trades one for the other.

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for ax, (index, label) in zip(axes.ravel(), enumerate(label_names)):
    matrix = confusion_matrix(y[:, index], (oof[:, index] >= tuned[index]).astype(int))
    sns.heatmap(matrix, annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["pred 0", "pred 1"], yticklabels=["true 0", "true 1"])
    ax.set_title(f"{label} (n+={int(y[:, index].sum())})", fontsize=10)
fig.suptitle("Confusion matrices at the tuned thresholds", y=1.01)
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "confusion_matrices.png", dpi=200, bbox_inches="tight")

## Reading the errors

Skim these and build a taxonomy for the report: quoted insults, sarcasm,
non-English comments, aggressive but non-toxic disagreement, annotator noise.

In [ ]:
from src.evaluation import error_examples

examples = error_examples(comment_text, y, oof, label_index=0,
                          threshold=tuned[0], n_examples=8)

print("=" * 30, "FALSE POSITIVES (clean, flagged as toxic)", "=" * 30)
for _, row in examples["false_positives"].iterrows():
    print(f"[{row.score:.3f}] {row.text[:180]}")

print("\n" + "=" * 30, "FALSE NEGATIVES (toxic, missed)", "=" * 30)
for _, row in examples["false_negatives"].iterrows():
    print(f"[{row.score:.3f}] {row.text[:180]}")

## Unintended bias

The model has seen that comments mentioning an identity term are often toxic,
and can end up penalising the term itself. The audit compares the false-positive
rate on the subgroup mentioning each term with the background rate: a ratio well
above 1 means a neutral sentence gets flagged for containing the word.

This is the finding worth building the discussion section around — and it is
what motivated Jigsaw's follow-up *Unintended Bias* competition.

In [ ]:
from src.evaluation import identity_term_report

bias = identity_term_report(comment_text, y, oof, label_index=0, threshold=tuned[0])
display(bias.round(4))

if not bias.empty:
    fig, ax = plt.subplots(figsize=(7.5, 4))
    ordered = bias.sort_values("fpr_ratio")
    ax.barh(ordered.index, ordered["fpr_ratio"], color="#C44E52")
    ax.axvline(1.0, color="black", lw=1)
    ax.text(1.03, -0.4, "no bias", fontsize=8)
    ax.set(xlabel="subgroup FPR / background FPR",
           title="False-positive rate on comments mentioning an identity term")
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(CONFIG.figures_dir / "identity_bias.png", dpi=200)

In [ ]:
# The concrete evidence: neutral sentences that the model flags anyway.
probe = [
    "i am a gay man and i edited this article",
    "she is a muslim scholar who wrote the source",
    "my black colleague suggested this citation",
    "the jewish community centre is mentioned in the article",
    "you are an idiot and your edits are garbage",
]
model = build_model("nbsvm", label_names=label_names).fit(comment_text, y)
pd.DataFrame(model.predict_proba(probe), columns=label_names, index=probe).round(3)

## Discussion points for the report

* Which labels are learnable and which are not — and whether the limit is the
  model or the ~200 positive examples the label actually has.
* The cost of binary relevance: does the model ever predict `severe_toxic`
  without `toxic`, an output the label hierarchy forbids?
* Whether the accuracy gap between the linear models and the transformer
  justifies the difference in training cost.
* Bias: which terms carry the highest false-positive ratio, and what would
  reduce it — reweighting, data augmentation with neutral sentences containing
  the term, or a fairness-aware objective.